# FAISS for TEXT - Quick Start
This notebook is a companion of chapter 2 of the "Domain Specific LLms in Action" book, author Guglielmo Iozzia, [Manning Publications](https://www.manning.com/), 2024.  
The code in this notebook is to introduce readers to the [FAISS](https://faiss.ai/index.html) library. No hardware acceleration required to execute all the code cells.  

Install the missing required packages in the Colab VM. Only FAISS for CPU is missing. Then downgrade the now available [SentenceTransformers](https://www.sbert.net/) release to 4.1.0 for compatibility with the code in this notebook.

In [1]:
!pip install faiss-cpu
!pip uninstall -y sentence-transformers
!pip install --no-input sentence-transformers==4.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 63.2 MB/s eta 0:00:00
Found existing installation: sentence-transformers 5.6.0
Uninstalling sentence-transformers-5.6.0:
  Successfully uninstalled sentence-transformers-5.6.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 117.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1
ERROR: pip's dependency resolver does not currently take into account all t

Import the necessary packages/classes.

In [2]:
"""Module to cluster embeddings and create indices."""
import faiss

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

Set the data corpus for this example and put it into a Pandas DataFrame.

In [3]:
data = [['His secret identity is Peter Parker', 'spiderman'],
        ['A businessman and engineer who ' +
         'runs the company Stark Industries',
         'ironman'],
        ['Superhuman spider-powers and abilities ' +
         'after being bitten by a radioactive spider',
         'spiderman'],
        ['A frail man enhanced to the peak of human ' +
         'physical perfection by an experimental super-soldier serum', 'captainamerica']
        ]
df = pd.DataFrame(data, columns = ['text', 'context'])

In [5]:
dataset = [("This movie is a masterpiece!", "Positive"),
("Not worth watching!", "Negative"),
("Terrific!", "Positive")
]

In [6]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [7]:
max_length = 128
formatted_data = [(f"[CLS] {text} [SEP]", label) for text, label in dataset]
tokenized_data = tokenizer(formatted_data,
padding=True,
truncation=True,
max_length=max_length,
return_tensors='pt')

In [8]:
import torch
from sklearn.preprocessing import LabelEncoder
input_ids = tokenized_data['input_ids']
attention_mask = tokenized_data['attention_mask']
labels = torch.tensor(LabelEncoder().fit_transform([label for _, label in dataset]))

In [10]:
from sklearn.model_selection import train_test_split
train_inputs, val_inputs, train_labels, val_labels,train_mask, val_mask = train_test_split(
input_ids, labels, attention_mask,
random_state=42, test_size=0.1
)

In [11]:
from torch.utils.data import Dataset
class CustomDataset(Dataset):
  def __init__(self, input_ids, attention_mask, labels):
    self.input_ids = input_ids
    self.attention_mask = attention_mask
    self.labels = labels
  def __len__(self):
    return len(self.input_ids)
  def __getitem__(self, idx):
    return {'input_ids': self.input_ids[idx],
    'attention_mask':
    self.attention_mask[idx],
    'labels': self.labels[idx]}

In [12]:
from torch.utils.data import DataLoader
batch_size = 4
train_dataset = CustomDataset(train_inputs,
train_mask, train_labels)
train_dataloader = DataLoader(train_dataset,
batch_size=batch_size,
shuffle=True)
val_dataset = CustomDataset(val_inputs,
val_mask, val_labels)
val_dataloader = DataLoader(val_dataset,
batch_size=batch_size,
shuffle=False)

In [14]:
from transformers import BertForSequenceClassification
model = BertForSequenceClassification.from_pretrained('bert-base-uncased',num_labels=len(set(labels)))


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
dataset = [("Once upon a time in a faraway swamp,",
"there lived an ugly ork."),
# More samples
]

In [16]:
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
formatted_data = [(f"[CLS] {context} [SEP] {target} [SEP]",) for context, target in dataset]
numerical_data = [tokenizer.encode(example[0],add_special_tokens=True)
for example in formatted_data]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Get embeddings from the data corpus, generate a FAISS index and add the embeddings to it (after normalization).  
To make the code in the cell below compatible with SentenceTransformers release 5.0+, simple replace the line  
```vectors = encoder.encode(text)```  
with  
```vectors = encoder.encode(text.to_list())```

In [ ]:
text = df['text']
encoder = SentenceTransformer("paraphrase-mpnet-base-v2")
vectors = encoder.encode(text)
vector_dimension = vectors.shape[1]
l2_index = faiss.IndexFlatL2(vector_dimension)
faiss.normalize_L2(vectors)
l2_index.add(vectors)

Prepare a search text to be used for similarity search with FAISS on the generated index.

In [ ]:
search_text = 'He throws webs'
search_vector = encoder.encode(search_text)
search_vector_as_array = np.array([search_vector])
faiss.normalize_L2(search_vector_as_array)

Perform a search within the created index (calculation of the distances between the search text and the strings within the index).

In [ ]:
k = l2_index.ntotal
distances, ann = l2_index.search(search_vector_as_array, k=k)

Prepare the results to be displayed in a user-friendly format.

In [ ]:
search_results = pd.DataFrame({'distances': distances[0], 'ann': ann[0]})
merged_df = pd.merge(search_results, df, left_on='ann', right_index=True)
merged_df.head()